# Etapa 1 — Diagnóstico e modelagem da situação atual

**Global Solution 2026.2 · Modelagem Matemática e Computacional · Mobilidade Urbana Inteligente**
**Integrantes:** Davi Queiroz Zuolo (RM 571669) · Gustavo Zagato Bottechia (RM 569420) · Daniel Vilela Mana (RM 571632)

Objetivo: recuperar quantos ônibus cada centro operacional (Norte, Sul, Leste, Oeste) possui, a partir dos indicadores consolidados do Relatório I.

## a) Variáveis
| Variável | Significado | Restrição |
|---|---|---|
| $N$ | ônibus alocados na região Norte | inteiro, $\ge 0$ |
| $S$ | ônibus alocados na região Sul | inteiro, $\ge 0$ |
| $L$ | ônibus alocados na região Leste | inteiro, $\ge 0$ |
| $O$ | ônibus alocados na região Oeste | inteiro, $\ge 0$ |

São quatro incógnitas porque há quatro centros e o que se perdeu foi exatamente a quantidade de cada um.

## b) Modelo matemático
Cada frase do Relatório I vira uma equação linear:

| Registro do relatório | Equação |
|---|---|
| A frota disponível é de 72 ônibus | $N + S + L + O = 72$ |
| Norte + Oeste + dobro do Sul = 76 | $N + 2S + O = 76$ |
| Sul + dobro do Leste = 52 | $S + 2L = 52$ |
| Norte + Leste + dobro do Oeste = 76 | $N + L + 2O = 76$ |

## c) Sistema matricial
Com $x = (N, S, L, O)^\top$, o sistema é $A\,x = b$:

$$
\begin{bmatrix} 1 & 1 & 1 & 1 \\ 1 & 2 & 0 & 1 \\ 0 & 1 & 2 & 0 \\ 1 & 0 & 1 & 2 \end{bmatrix}
\begin{bmatrix} N \\ S \\ L \\ O \end{bmatrix} =
\begin{bmatrix} 72 \\ 76 \\ 52 \\ 76 \end{bmatrix}
$$

Cada linha de $A$ é um registro; cada coluna, uma região. O coeficiente $a_{ij}$ diz quantas vezes a frota da região $j$ entra no registro $i$.

In [1]:
import numpy as np

regioes = ["Norte", "Sul", "Leste", "Oeste"]
A = np.array([[1, 1, 1, 1],
              [1, 2, 0, 1],
              [0, 1, 2, 0],
              [1, 0, 1, 2]], dtype=float)
b = np.array([72, 76, 52, 76], dtype=float)
print("A =\n", A, "\n\nb =", b)

A =
 [[1. 1. 1. 1.]
 [1. 2. 0. 1.]
 [0. 1. 2. 0.]
 [1. 0. 1. 2.]] 

b = [72. 76. 52. 76.]


## d) As informações determinam uma única distribuição?
Um sistema linear $Ax=b$ com $n$ incógnitas pode ter nenhuma, uma ou infinitas soluções. Dois critérios equivalentes decidem:

1. **Determinante.** Se $A$ é quadrada e $\det(A) \neq 0$, então $A$ é invertível e a solução é única: $x = A^{-1}b$.
2. **Teorema de Rouché–Capelli.** O sistema tem solução se e só se $\text{posto}(A) = \text{posto}([A\,|\,b])$. Se esse posto comum for igual ao número de incógnitas, a solução é única.

Interpretação: posto 4 significa que os quatro registros são **linearmente independentes**, isto é, nenhum deles é combinação dos outros. Cada registro traz informação nova, e quatro informações independentes fixam quatro incógnitas.

In [2]:
det = np.linalg.det(A)
posto_A = np.linalg.matrix_rank(A)
posto_Ab = np.linalg.matrix_rank(np.column_stack([A, b]))
print(f"det(A) = {det:.4f}")
print(f"posto(A) = {posto_A} | posto([A|b]) = {posto_Ab} | incógnitas = {A.shape[1]}")
assert abs(det) > 1e-9 and posto_A == posto_Ab == A.shape[1]
print("Conclusão: det(A) != 0 e posto(A) = posto([A|b]) = 4 -> sistema possível e determinado (solução única).")

det(A) = 3.0000
posto(A) = 4 | posto([A|b]) = 4 | incógnitas = 4
Conclusão: det(A) != 0 e posto(A) = posto([A|b]) = 4 -> sistema possível e determinado (solução única).


O determinante também pode ser conferido à mão, pela expansão de Laplace na primeira coluna (que tem um zero):

$$\det(A) = 1\cdot\begin{vmatrix}2&0&1\\1&2&0\\0&1&2\end{vmatrix} - 1\cdot\begin{vmatrix}1&1&1\\1&2&0\\0&1&2\end{vmatrix} + 0 - 1\cdot\begin{vmatrix}1&1&1\\2&0&1\\1&2&0\end{vmatrix} = 9 - 3 - 3 = 3$$

## e) Resolução computacional
`np.linalg.solve` resolve $Ax=b$ por decomposição LU com pivoteamento, que é mais estável e mais barato do que calcular $A^{-1}$ explicitamente.

In [3]:
x = np.linalg.solve(A, b)
for nome, valor in zip(regioes, x):
    print(f"{nome:6s}: {valor:.0f} ônibus")
print("Total :", f"{x.sum():.0f}")

Norte : 12 ônibus
Sul   : 20 ônibus
Leste : 16 ônibus
Oeste : 24 ônibus
Total : 72


## f) Verificação
Três checagens: (1) a solução satisfaz os quatro registros, isto é, $Ax = b$; (2) as quantidades são inteiras; (3) não há valores negativos. As duas últimas são exigências do enunciado que o sistema linear, sozinho, não garante: por isso são verificadas depois.

In [4]:
residuo = A @ x - b
rotulos = ["frota total = 72", "N + 2S + O = 76", "S + 2L = 52", "N + L + 2O = 76"]
for rotulo, obtido, esperado in zip(rotulos, A @ x, b):
    print(f"{rotulo:18s} -> obtido {obtido:.0f} | esperado {esperado:.0f} | {'OK' if np.isclose(obtido, esperado) else 'FALHOU'}")
assert np.allclose(residuo, 0), "a solução não satisfaz os registros"
assert np.allclose(x, np.rint(x)), "há quantidade não inteira"
assert (x >= 0).all(), "há quantidade negativa"
print("\nInteiras: sim | Não negativas: sim | Resíduo máximo:", np.abs(residuo).max())

frota total = 72   -> obtido 72 | esperado 72 | OK
N + 2S + O = 76    -> obtido 76 | esperado 76 | OK
S + 2L = 52        -> obtido 52 | esperado 52 | OK
N + L + 2O = 76    -> obtido 76 | esperado 76 | OK

Inteiras: sim | Não negativas: sim | Resíduo máximo: 0.0


## Conclusão da Etapa 1
A distribuição atual da frota é **Norte = 12, Sul = 20, Leste = 16, Oeste = 24** (total 72). Ela é a única compatível com os registros, porque a matriz dos coeficientes tem determinante 3 (diferente de zero) e posto 4. A solução satisfaz os quatro registros e respeita as restrições de integralidade e não negatividade.

Este vetor é a entrada da Etapa 2.